<a href="https://colab.research.google.com/github/sauravv19/A-Security-Analysis-of-Tf-Idf-And-Deep-Learning-Model-for-Hate-Speech-Detection-in-YouTube/blob/main/Hate%20Speech%20Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yt-dlp

In [ ]:
import torch
import numpy as np
from scipy.special import softmax
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline, AutoTokenizer, AutoModelForSequenceClassification, RobertaTokenizerFast, RobertaForSequenceClassification
from transformers.models.whisper.processing_whisper import WhisperProcessor
from transformers.models.whisper.modeling_whisper import WhisperForConditionalGeneration
from transformers.pipelines.automatic_speech_recognition import AutomaticSpeechRecognitionPipeline
from subprocess import run, PIPE
from typing import Self
from uuid import uuid4
from os import makedirs, remove

In [ ]:
MAX_SEQUENCE_LENGTH = 512
OVERLAP = 50
STRIDE = MAX_SEQUENCE_LENGTH - OVERLAP
MAX_NEW_TOKENS = 128
CHUNK_LENGTH = 30
VIDEO_URL = "https://youtu.be/U1S8bduxLKM?si=8EIVyKJHOeqZR5qu"

In [ ]:
class ASRHate:

    def __pipe__(self) -> Self:
      self.__asr_pipe__ = pipeline(
          "automatic-speech-recognition",
          model=self.__wh_model__,
          tokenizer=self.__wh_processor__.tokenizer,
          feature_extractor=self.__wh_processor__.feature_extractor,
          chunk_length_s=CHUNK_LENGTH,
          device=self.device,
          generate_kwargs={"max_new_tokens": MAX_NEW_TOKENS},
      )
      return self

    def __on_device__(self) -> Self:
      self.device = "cuda" if torch.cuda.is_available() else "cpu"
      self.__wh_model__.to(self.device)
      return self

    @staticmethod
    def __ytdlp__(video_url: str) -> str:
      makedirs("/content/outputs", exist_ok=True)
      out = f"/content/outputs/output-{str(uuid4())}.wav"
      proc = run(f'yt-dlp --extract-audio --audio-format wav --output {out} {video_url}', shell=True, stderr=PIPE, stdout=PIPE)
      if proc.returncode == 0:
        return out
      raise Exception(proc.stderr.decode(errors="ignore"))

    def transcribe(self, video_url: str) -> Self:
      self.__audio_path__ = ASRHate.__ytdlp__(video_url)
      self.transcription = self.__asr_pipe__(self.__audio_path__)
      return self

    def classify(self) -> Self:
      inputs = self.__classifier_tokenizer__(
          self.transcription["text"],
          max_length=MAX_SEQUENCE_LENGTH,
          padding="max_length",
          truncation=True,
          return_overflowing_tokens=True,
          stride=STRIDE,
          return_tensors="pt"
      )
      model_inputs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            # Add "token_type_ids": inputs["token_type_ids"].to(self.device) if your tokenizer returns them
      }
      with torch.no_grad():
        outputs = self.__classifier_model__(**model_inputs)
      if self.device == 'cuda':
        logits_np = outputs.logits.cpu().numpy()
      else:
        logits_np = outputs.logits.numpy()

      self.probabilities = softmax(logits_np, axis=1)
      return self

    def clean(self) -> None:
      remove(self.__audio_path__)
      print("removed", self.__audio_path__)
      delattr(self, "__audio_path__")

    def chunkwise(self):
      percentages = self.probabilities * 100
      percentages_list = percentages.tolist()
      print(percentages_list)
      label_0 = self.__classifier_model__.config.id2label[0].title()
      label_1 = self.__classifier_model__.config.id2label[1].title()
      for i, chunk_percentages in enumerate(percentages_list):
        print(f"Chunk {i}: {label_0} {chunk_percentages[0]:.2f}% {label_1} {chunk_percentages[1]:.2f}%")


    def overall(self) -> None:
      mean_probabilities_np = np.mean(self.probabilities, axis=0)
      mean_percentages_np = mean_probabilities_np * 100
      mean_percentages_list = mean_percentages_np.tolist()
      # Print the mean percentage for each label from the list
      for j, mean_percentage in enumerate(mean_percentages_list):
          label = self.__classifier_model__.config.id2label[j]
          print(f"Mean {label.title()}: {mean_percentage:.2f}%")



    def __init__(self):
      self.__wh_processor__ = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo")
      self.__wh_model__ = WhisperForConditionalGeneration.from_pretrained(
          "openai/whisper-large-v3-turbo"
      )
      # Use AutoTokenizer for robustness
      self.__classifier_tokenizer__ = AutoTokenizer.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
      self.__classifier_model__ = RobertaForSequenceClassification.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
      self.__on_device__().__pipe__()

In [ ]:
asr_hate = ASRHate()

In [ ]:
asr_hate.transcribe(VIDEO_URL).classify().clean()

In [ ]:
print(asr_hate.transcription['text'])

In [ ]:
asr_hate.chunkwise()

In [ ]:
asr_hate.overall()